# Phase 6 DAEAC FCBA RR Late-Fusion NSV Weight Grid

Runs three 3-class `N/S/V` pretrain experiments in one Kaggle notebook: sqrt class weights, capped inverse weights, and S-weight scaling.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import numpy as np
import pandas as pd
import yaml
os.environ['PYTHONUNBUFFERED'] = '1'

REPO_URL = 'https://github.com/YOUR_ORG/Best-thesis-in-council.git'
BRANCH = 'main'
BASE_CONFIG = Path('configs/phase6_daeac_fcba_latefusion_rr_nsv.yaml')
GENERATED_CONFIG_DIR = Path('configs/generated')

WORK = Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'
ECG = REPO / 'ecg_thesis'
print('Repo path:', ECG)

## 1. Clone Repo and Install

In [ ]:
if not ECG.exists():
    assert 'YOUR_ORG' not in REPO_URL, 'Edit REPO_URL before cloning.'
    !git clone --branch {BRANCH} {REPO_URL} /kaggle/working/Best-thesis-in-council
else:
    print('Repo already exists:', ECG)
assert ECG.exists(), ECG
os.chdir(ECG)
!pip install -q -r requirements.txt
!python scripts/check_repo.py

## 2. Copy NSV RR Bundle

In [ ]:
candidate_dirs = [p.parent for p in Path('/kaggle/input').glob('**/record_split_manifest.json')]
print('Candidate record-split bundle dirs:')
for p in candidate_dirs:
    print(' -', p)
assert len(candidate_dirs) == 1, f'Expected one NSV record-split bundle, found {candidate_dirs}'

DEST = Path('data/processed/phase6_daeac_record_splits_nsv')
DEST.mkdir(parents=True, exist_ok=True)
for src in sorted(candidate_dirs[0].glob('*.npz')):
    shutil.copy2(src, DEST / src.name)
manifest = candidate_dirs[0] / 'record_split_manifest.json'
if manifest.exists():
    shutil.copy2(manifest, DEST / manifest.name)

for name in ('ds1_train.npz', 'ds1_val.npz', 'ds2_test.npz'):
    path = DEST / name
    assert path.exists(), path
    with np.load(path, allow_pickle=True) as data:
        assert data['class_names'].tolist() == ['N', 'S', 'V'], (name, data['class_names'].tolist())
        assert 'rr_features' in data.files, f'{name} has no rr_features.'
        assert data['rr_features'].shape == (len(data['x']), 7), (name, data['rr_features'].shape)
        if 'y' in data.files:
            assert set(np.unique(data['y']).tolist()).issubset({0, 1, 2}), (name, np.unique(data['y']).tolist())

!ls -lh data/processed/phase6_daeac_record_splits_nsv

## 3. Create Three Generated Configs

In [ ]:
base_cfg = yaml.safe_load(BASE_CONFIG.read_text())
GENERATED_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

RUNS = [
    {
        'run_id': 'sqrtw',
        'label': 'sqrt class weight',
        'training_updates': {'class_weight_mode': 'sqrt'},
    },
    {
        'run_id': 'cap5',
        'label': 'inverse class weight capped at 5',
        'training_updates': {'class_weight_mode': 'inverse', 'class_weight_cap': 5.0},
    },
    {
        'run_id': 's025',
        'label': 'inverse class weight with S scale 0.25',
        'training_updates': {'class_weight_mode': 'inverse', 'class_weight_scales': {'S': 0.25}},
    },
]

generated = []
for run in RUNS:
    cfg = json.loads(json.dumps(base_cfg))
    run_id = run['run_id']
    output_dir = f'outputs/phase6_daeac_fcba_latefusion_rr_nsv_{run_id}'
    base_prefix = f'daeac_fcba_rr_nsv_{run_id}_base'
    uda_prefix = f'daeac_fcba_rr_nsv_{run_id}_uda'

    cfg['logging']['wandb']['group'] = f'phase6_daeac_fcba_latefusion_rr_nsv_{run_id}'
    cfg['logging']['wandb']['tags'] = ['phase6', 'daeac', 'fcba', 'rr_late_fusion', 'nsv', run_id]
    cfg['paths']['output_dir'] = output_dir
    cfg['training']['checkpoint_prefix'] = base_prefix
    cfg['training']['epochs'] = 50
    cfg['training'].update(run['training_updates'])
    cfg['adaptation']['checkpoint_prefix'] = uda_prefix
    cfg['adaptation']['epochs'] = 50
    cfg['adaptation']['init_checkpoint'] = f'{output_dir}/checkpoints/{base_prefix}_best.pt'
    cfg['adaptation'].update(run['training_updates'])

    cfg_path = GENERATED_CONFIG_DIR / f'phase6_daeac_fcba_latefusion_rr_nsv_{run_id}.yaml'
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    run['config'] = str(cfg_path)
    run['output_dir'] = output_dir
    run['method_name'] = f'{base_prefix}_best'
    generated.append(run)
    print(run_id, cfg_path, run['training_updates'])

print('Generated configs:', [r['config'] for r in generated])

## 4. Validate Once

In [ ]:
subprocess.run([sys.executable, 'scripts/phase6_daeac_paper/00_validate_data.py', '--config', generated[0]['config']], check=True)

## 5. Run All Three Pretrains and DS2 Evaluations

In [ ]:
results = []
for run in generated:
    print('\n' + '=' * 80)
    print('RUN', run['run_id'], '-', run['label'])
    print('=' * 80)
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_paper/01_train_base.py',
        '--config', run['config'],
    ], check=True)

    summary_path = Path(run['output_dir']) / 'metrics' / 'daeac_base_train_summary.json'
    summary = json.loads(summary_path.read_text())
    best_checkpoint = Path(summary['best_checkpoint'])
    assert best_checkpoint.exists(), best_checkpoint

    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_paper/03_eval.py',
        '--config', run['config'],
        '--checkpoint', str(best_checkpoint),
        '--method-name', run['method_name'],
        '--dataset', 'target',
    ], check=True)

    metrics_path = Path(run['output_dir']) / 'metrics' / f"{run['method_name']}_target_test_metrics.json"
    metrics = json.loads(metrics_path.read_text())
    cm = np.asarray(metrics['confusion_matrix'])
    results.append({
        'run_id': run['run_id'],
        'label': run['label'],
        'best_epoch': summary['best_epoch'],
        'best_val_macro_f1': summary['best_val_macro_f1'],
        'test_accuracy': metrics['accuracy'],
        'test_macro_f1': metrics['macro_f1'],
        'N_f1': metrics['per_class']['N']['f1'],
        'S_precision': metrics['per_class']['S']['precision'],
        'S_recall': metrics['per_class']['S']['recall'],
        'S_f1': metrics['per_class']['S']['f1'],
        'V_f1': metrics['per_class']['V']['f1'],
        'N_to_S': int(cm[0, 1]),
        'S_to_N': int(cm[1, 0]),
        'S_to_V': int(cm[1, 2]),
        'pred_S': int(cm[:, 1].sum()),
        'summary_path': str(summary_path),
        'metrics_path': str(metrics_path),
    })

df = pd.DataFrame(results).sort_values('test_macro_f1', ascending=False)
display(df)
summary_csv = Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv_weight_grid_summary.csv')
summary_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(summary_csv, index=False)
print('Saved', summary_csv)

## 6. Package Outputs

In [ ]:
!zip -r /kaggle/working/phase6_daeac_fcba_latefusion_rr_nsv_weight_grid_outputs.zip outputs/phase6_daeac_fcba_latefusion_rr_nsv_* outputs/phase6_daeac_fcba_latefusion_rr_nsv_weight_grid_summary.csv configs/generated